# RZSM Evaluation — Paper Results

Trains all baselines, evaluates across disturbance scenarios, and generates
LaTeX-ready tables and matplotlib figures for the paper.

**Methods:** Vanilla DDPG, RARL, SA-MDP, Domain Randomization, RZSM (ours)

**Scenarios:** Nominal, External Force, Parameter Perturbation, Observation Noise, Combined

## 1. Setup

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.size': 11,
    'font.family': 'serif',
    'figure.dpi': 150,
    'savefig.bbox': 'tight',
})

# Ensure project root is on path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')

## 2. Configuration

In [ ]:
ENVS = ['Ant-v5', 'HalfCheetah-v5', 'Walker2d-v5', 'Humanoid-v5']
METHODS = ['vanilla', 'rarl', 'sa_mdp', 'dr', 'rzsm']
SCENARIOS = ['nominal', 'force', 'params', 'noise', 'combined']

NUM_ENVS = 10        # parallel envs for training
EPOCHS = 100         # training epochs per method
EVAL_EPISODES = 50   # evaluation episodes per (method, scenario)
CHECKPOINT_DIR = 'logs'
RESULTS_DIR = 'results'

# For quick testing, reduce these:
# ENVS = ['Ant-v5']
# EPOCHS = 5
# EVAL_EPISODES = 10

## 3. Train All Methods

Skip cells for methods that already have checkpoints.

In [ ]:
def ckpt_exists(method, env):
    env_tag = env.replace('-', '_').lower()
    path = f'{CHECKPOINT_DIR}/{method}/checkpoints'
    return os.path.isdir(path)

for m in METHODS:
    print(f'{m}: {"exists" if ckpt_exists(m, ENVS[0]) else "NOT FOUND"}')

In [ ]:
# ── Vanilla DDPG ──
if not ckpt_exists('vanilla', ENVS[0]):
    !python -m src.train --env {ENVS[0]} --mode nominal --num-envs {NUM_ENVS} \
        --epochs {EPOCHS} --log-dir logs/vanilla
else:
    print('Vanilla checkpoints found, skipping.')

In [ ]:
# ── RARL (adversarial, no transformer) ──
if not ckpt_exists('rarl', ENVS[0]):
    !python -m src.train --env {ENVS[0]} --mode adversarial --no-transformer \
        --num-envs {NUM_ENVS} --epochs {EPOCHS} --log-dir logs/rarl
else:
    print('RARL checkpoints found, skipping.')

In [ ]:
# ── SA-MDP ──
if not ckpt_exists('sa_mdp', ENVS[0]):
    !python -m src.baselines.sa_mdp --env {ENVS[0]} --num-envs {NUM_ENVS} \
        --epochs {EPOCHS} --log-dir logs/sa_mdp
else:
    print('SA-MDP checkpoints found, skipping.')

In [ ]:
# ── Domain Randomization ──
if not ckpt_exists('dr', ENVS[0]):
    !python -m src.baselines.domain_randomization --env {ENVS[0]} --num-envs {NUM_ENVS} \
        --epochs {EPOCHS} --log-dir logs/dr
else:
    print('DR checkpoints found, skipping.')

In [ ]:
# ── RZSM (ours: adversarial + transformer) ──
if not ckpt_exists('rzsm', ENVS[0]):
    !python -m src.train --env {ENVS[0]} --mode adversarial \
        --num-envs {NUM_ENVS} --epochs {EPOCHS} \
        --pi-opt-path logs/vanilla/checkpoints \
        --log-dir logs/rzsm
else:
    print('RZSM checkpoints found, skipping.')

## 4. Run Evaluation

In [ ]:
methods_str = ','.join(METHODS)
scenarios_str = ','.join(SCENARIOS)

!python -m src.eval \
    --env {ENVS[0]} \
    --methods {methods_str} \
    --scenarios {scenarios_str} \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --episodes {EVAL_EPISODES} \
    --output {RESULTS_DIR}

## 5. Table 1 — Main Comparison (LaTeX)

In [ ]:
env_tag = ENVS[0].replace('-', '_')
csv_path = f'{RESULTS_DIR}/{env_tag}/comparison_table.csv'
df = pd.read_csv(csv_path)

# Pivot: methods as rows, scenarios as columns
df['result_str'] = df.apply(
    lambda r: f"${r['mean_return']:.0f} \\pm {r['std_return']:.0f}$", axis=1
)
pivot = df.pivot(index='method', columns='scenario', values='result_str')
pivot = pivot.reindex(index=METHODS, columns=SCENARIOS)

# Pretty names
name_map = {'vanilla': 'Vanilla DDPG', 'rarl': 'RARL', 'sa_mdp': 'SA-MDP',
            'dr': 'Domain Rand.', 'rzsm': 'RZSM (Ours)'}
pivot.index = [name_map.get(m, m) for m in pivot.index]
pivot.columns = [c.capitalize() for c in pivot.columns]

print('\n% LaTeX table — paste into paper.tex')
print(pivot.to_latex(escape=False, column_format='l' + 'c' * len(SCENARIOS)))
display(pivot)

## 6. Figure 1 — Robustness Comparison (Bar Chart)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

df_plot = df.copy()
n_methods = len(METHODS)
n_scenarios = len(SCENARIOS)
x = np.arange(n_scenarios)
width = 0.8 / n_methods

colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3']
labels = ['Vanilla DDPG', 'RARL', 'SA-MDP', 'Domain Rand.', 'RZSM (Ours)']

for i, method in enumerate(METHODS):
    subset = df_plot[df_plot['method'] == method].set_index('scenario')
    subset = subset.reindex(SCENARIOS)
    means = subset['mean_return'].values
    stds = subset['std_return'].values
    ax.bar(x + i * width - 0.4 + width/2, means, width,
           yerr=stds, label=labels[i], color=colors[i], alpha=0.85,
           capsize=2, error_kw={'linewidth': 0.8})

ax.set_xticks(x)
ax.set_xticklabels([s.capitalize() for s in SCENARIOS])
ax.set_ylabel('Mean Episode Return')
ax.set_title(f'Robustness Comparison — {ENVS[0]}')
ax.legend(loc='upper right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig1_robustness_comparison.pdf')
plt.show()

## 7. Figure 2 — Training Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for i, method in enumerate(METHODS):
    csv_file = f'{CHECKPOINT_DIR}/{method}/training_returns.csv'
    if not os.path.isfile(csv_file):
        continue
    data = pd.read_csv(csv_file)
    # Smooth with rolling window
    col = 'episode_return' if 'episode_return' in data.columns else data.columns[1]
    smoothed = data[col].rolling(window=20, min_periods=1).mean()
    ax.plot(data.iloc[:, 0], smoothed, label=labels[i], color=colors[i],
            linewidth=1.2, alpha=0.9)

ax.set_xlabel('Environment Steps')
ax.set_ylabel('Episode Return (smoothed)')
ax.set_title(f'Training Curves — {ENVS[0]}')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig2_training_curves.pdf')
plt.show()

## 8. Figure 3 — Robustness Ratio Heatmap

In [ ]:
# Compute robustness ratio: J_scenario / J_nominal
nominal = df[df['scenario'] == 'nominal'].set_index('method')['mean_return']
df_ratio = df.copy()
df_ratio['ratio'] = df_ratio.apply(
    lambda r: r['mean_return'] / nominal[r['method']]
    if r['method'] in nominal.index and nominal[r['method']] != 0
    else float('nan'), axis=1
)

ratio_pivot = df_ratio.pivot(index='method', columns='scenario', values='ratio')
ratio_pivot = ratio_pivot.reindex(index=METHODS,
                                   columns=[s for s in SCENARIOS if s != 'nominal'])
ratio_pivot.index = [name_map.get(m, m) for m in ratio_pivot.index]

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(ratio_pivot.values, cmap='RdYlGn', vmin=0, vmax=1.2, aspect='auto')
ax.set_xticks(range(len(ratio_pivot.columns)))
ax.set_xticklabels([c.capitalize() for c in ratio_pivot.columns])
ax.set_yticks(range(len(ratio_pivot.index)))
ax.set_yticklabels(ratio_pivot.index)

# Annotate cells
for i in range(len(ratio_pivot.index)):
    for j in range(len(ratio_pivot.columns)):
        val = ratio_pivot.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=10,
                    color='black' if 0.3 < val < 0.9 else 'white')

plt.colorbar(im, ax=ax, label='$J_{rob} / J_{nom}$')
ax.set_title(f'Robustness Ratio — {ENVS[0]}')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/fig3_robustness_ratio.pdf')
plt.show()

# Print LaTeX table
print('\n% Robustness ratio table')
print(ratio_pivot.applymap(lambda x: f'{x:.2f}' if not np.isnan(x) else '--').to_latex())

## 9. Detection Metrics (RZSM Only)

In [ ]:
import json

det_path = f'{RESULTS_DIR}/{env_tag}/detection_metrics.json'
if os.path.isfile(det_path):
    with open(det_path) as f:
        det = json.load(f)
    print('Disturbance Detection Metrics (RZSM):')
    print(json.dumps(det, indent=2))
else:
    print('No detection metrics found. Run RZSM evaluation first.')

## 10. Ablation Study

In [ ]:
# Ablation: compare RZSM variants
# Requires training additional models:
#   - rzsm_no_det:  AdversarialDDPGAgent with use_transformer=False
#   - rzsm_L8:      AdversarialDDPGAgent with transformer_seq_len=8
#   - rzsm_L16:     transformer_seq_len=16
#   - rzsm_L32:     transformer_seq_len=32
#   - rzsm_L64:     transformer_seq_len=64

ablation_variants = {
    'RZSM (full, L=20)': 'rzsm',
    'No detector':       'rarl',       # same as RARL
    # Add more after training:
    # 'L=8':  'rzsm_L8',
    # 'L=16': 'rzsm_L16',
    # 'L=32': 'rzsm_L32',
    # 'L=64': 'rzsm_L64',
}

ablation_rows = []
for label, method in ablation_variants.items():
    subset = df[df['method'] == method]
    if len(subset) == 0:
        continue
    for _, row in subset.iterrows():
        ablation_rows.append({
            'Variant': label,
            'Scenario': row['scenario'].capitalize(),
            'Return': f"{row['mean_return']:.0f} +/- {row['std_return']:.0f}",
        })

if ablation_rows:
    abl_df = pd.DataFrame(ablation_rows)
    abl_pivot = abl_df.pivot(index='Variant', columns='Scenario', values='Return')
    display(abl_pivot)
    print('\n% Ablation LaTeX table')
    print(abl_pivot.to_latex())
else:
    print('No ablation data. Train additional RZSM variants first.')

## 11. Export Figures

In [ ]:
import glob

pdfs = glob.glob(f'{RESULTS_DIR}/*.pdf')
print(f'Saved {len(pdfs)} figures:')
for p in sorted(pdfs):
    print(f'  {p}')

csvs = glob.glob(f'{RESULTS_DIR}/**/*.csv', recursive=True)
print(f'\nSaved {len(csvs)} data files:')
for p in sorted(csvs):
    print(f'  {p}')